# 🏛️ 01_pipeline_execution.ipynb
### MacroRisk Weekly Intelligence Pipeline (v8.6 Enterprise)
**Master Production Execution Workbench**
- Executes the 4-Stage LangGraph StateGraph DAG.
- Ingests live Fiscal Data REST APIs and real-time Factor ETF spreads.
- Generates the C-Suite Markdown Report and 7 BCBS 239 CSV appendices.

In [ ]:
"""
CELL 1: Environment Setup & Google Drive Storage Mount

Input:
- Target Google Drive mount directory (/content/drive/MyDrive/Colab Notebooks/macro_risk_pipeline)

Output:
- Mounted Google Drive with verified persistence folders (artifacts/, error_logs/, Production/)
- Verified active working directory pointing to repository root (/content/macro_risk_pipeline)

Processing Logic:
1. Connects to Google Drive to ensure persistent storage across ephemeral Colab sessions.
2. Sets GOOGLE_DRIVE_MOUNT_PATH environment variable for BCBS 239 data isolation.
3. Ensures all subdirectories exist and changes working directory to repository root.
"""

# Import os module to manage directories and environment variables
import os
# Import sys module for system interpreter references
import sys
# Import drive module from google.colab to mount Google Drive filesystem
from google.colab import drive

# Mount Google Drive filesystem without force remounting
drive.mount('/content/drive', force_remount=False)

# Define target Google Drive storage folder for all pipeline artifacts
drive_path = '/content/drive/MyDrive/Colab Notebooks/macro_risk_pipeline'

# Ensure persistent artifacts directory exists in Google Drive
os.makedirs(f'{drive_path}/artifacts', exist_ok=True)
# Ensure persistent error logs directory exists in Google Drive
os.makedirs(f'{drive_path}/error_logs', exist_ok=True)
# Ensure persistent Production reports directory exists in Google Drive
os.makedirs(f'{drive_path}/Production', exist_ok=True)

# Set environment variable so all pipeline stages save directly to Google Drive
os.environ['GOOGLE_DRIVE_MOUNT_PATH'] = drive_path

# Define repository directory path on local disk
repo_path = '/content/macro_risk_pipeline'
# Switch working directory to repository root if folder exists
if os.path.exists(repo_path):
    # Change directory
    os.chdir(repo_path)

# Print environment confirmation summary
print('=' * 60)
print('ENVIRONMENT MOUNTED & CONFIGURED SUCCESSFULLY!')
print(f'Active Working Directory: {os.getcwd()}')
print(f'Google Drive Persistence: {drive_path}')
print('=' * 60)


In [ ]:
"""
CELL 2: Run Full Automated Test Suite (All 36 Pytest Tests)

Input:
- Test suites in tests/ covering schemas, math utils, scrapers, stages, and LangGraph DAG

Output:
- Detailed test execution results with exit code (Exit code: 0 = All 36 tests passed)

Processing Logic:
1. Invokes pytest runner directly from Python on the tests/ directory.
2. Validates math formulas, mock LLM parsing, v14.6 unknown rules, and LangGraph state machine.
"""

# Import pytest to run automated unit and integration tests directly in Python
import pytest

# Execute pytest directly on tests/ directory with verbose output and stdout capture
exit_code = pytest.main(['-v', '-s', 'tests/'])

# Print human-readable summary based on exit code
if exit_code == 0:
    # Print success confirmation
    print('\nALL 36 REPOSITORY TESTS PASSED PERFECTLY (Exit Code: 0)')
else:
    # Print failure warning
    print(f'\nTESTS FAILED (Exit Code: {exit_code})')


In [ ]:
"""
CELL 3: Execute Master LangGraph StateGraph Pipeline

Input:
- System datetime localized to US Eastern Time
- Google Gemini API Key from Colab Secrets vault (GEMINI_API_KEY)
- Direct U.S. Fiscal Data REST APIs & yfinance factor ETF feeds

Output:
- Validated Stage4FormatterOutput Pydantic object
- Persisted report in Production/MacroRisk_Weekly_Intelligence_Report_{DATE}.md
- 7 Standalone CSV files in Production/csv_appendices/

Processing Logic:
1. Invokes run_pipeline() from src/graph.py executing the 5-node LangGraph StateGraph.
2. Runs Temporal Engine (S1) -> Harvester (S2) -> Quant Engine (S3) -> Formatter (S4) -> Fact-Auditor (S5).
3. Automatically saves all production artifacts to mounted Google Drive.
"""

# Import master pipeline runner from graph coordinator module
from src.graph import run_pipeline

# Execute the complete 4-Stage LangGraph pipeline DAG with artifact persistence
final_output = run_pipeline(save_artifacts=True)

# Print structured execution summary header
print('\n' + '=' * 80)
print('PIPELINE DAG EXECUTION SUMMARY:')
print('=' * 80)
# Print path to generated report
print(f'Report Path:      {final_output.report_file_path}')
# Print executive briefing word count
print(f'Briefing Words:   {final_output.word_count_briefing} (Target <= 200 words)')
# Print count of exported CSV appendices
print(f'CSV Appendices:   {len(final_output.csv_file_paths)} files saved to Google Drive')
# Print individual CSV file paths
for csv_path in final_output.csv_file_paths:
    print(f'  - {csv_path}')
print('=' * 80)


In [ ]:
"""
CELL 4: Preview C-Suite Markdown Production Report

Input:
- Path to generated production markdown report (final_output.report_file_path)

Output:
- Formatted console preview of Sections 1 through 5 of the generated report

Processing Logic:
1. Opens the persisted report file from Google Drive.
2. Reads and displays the first 2,500 characters covering Executive Briefing, Risk Matrix, and Calendar.
"""

# Open the persisted report file in read mode with UTF-8 encoding
with open(final_output.report_file_path, 'r', encoding='utf-8') as f:
    # Read entire report text content
    report_content = f.read()

# Print preview header banner
print('=' * 80)
print('PRODUCTION REPORT PREVIEW (SECTIONS 1 - 5):')
print('=' * 80)
# Display preview excerpt of report
print(report_content[:2500])
print('\n... [REMAINDER OF REPORT & 7 CSV BLOCKS SAVED IN GOOGLE DRIVE] ...')
print('=' * 80)


In [ ]:
"""
CELL 5: Inspect Live Data Ingestion & Real-Time Factor Spreads

Input:
- Public REST endpoint: api.fiscaldata.treasury.gov
- Real-time ETF quotes: MTUM, VLUE, QUAL, USMV, IWM, SPY via yfinance

Output:
- Formatted display of verified live TGA balance and Treasury auctions
- Real-time Asness factor spread differentials and quantitative regime states

Processing Logic:
1. Calls collect_live_public_macro_data() to query U.S. Treasury Fiscal Data REST APIs directly.
2. Calls fetch_factor_etf_spreads() to compute rolling 5-day return spreads on factor ETF pairs.
3. Displays all retrieved values with attached [VERIFIED_OFFICIAL] epistemic markers.
"""

# Import direct Fiscal Data collector function
from src.data.public_macro_api import collect_live_public_macro_data
# Import factor ETF spread harvester function
from src.data.factor_scraper import fetch_factor_etf_spreads

# Execute live U.S. Fiscal Data REST query
live_plumbing, live_auctions = collect_live_public_macro_data()

# Print Fiscal Data header
print('=' * 70)
print('LIVE FISCAL DATA REST INGESTION (api.fiscaldata.treasury.gov):')
print('=' * 70)
# Iterate through plumbing metrics
for key, val in live_plumbing.items():
    print(f'  {key}: {val}')
print(f'\nRetrieved {len(live_auctions)} Live Treasury Auctions:')
# Iterate through live auctions
for auc in live_auctions:
    print(f'  - [{auc.auction_date}] {auc.term} {auc.security_type}: {auc.offering_size_usd}')

# Execute real-time factor ETF spread calculation
live_factors = fetch_factor_etf_spreads()

# Print factor spreads header
print('\n' + '=' * 70)
print('REAL-TIME FACTOR ETF ROTATION SPREADS (yfinance):')
print('=' * 70)
# Iterate through factor regimes
for f in live_factors:
    print(f'  - {f.factor_pair}: {f.regime_state}')
    print(f'    Observation: {f.spread_observation}')
print('=' * 70)
